# Project 1: one backbone, three vision tasks

Individual project, weeks 5–9. Train your own CNN from random initialization
for 200-class classification. Lock the selected backbone, then train detection
and segmentation branches with its architecture, weights, and buffers frozen.
External pretrained weights and external training data are prohibited.

Coding agents are allowed. Document their use and explain your design choices.
The default cells perform no training or downloads. Complete the analysis
sections and submit an executed copy as `report.ipynb`.

## 1. Environment and paths

Install the dependencies in `requirements.txt` and start Jupyter in the
extracted starter directory. It contains `public/` and `src/`; the downloader
creates `data/`.

For final replay, extract your submission into `submissions/YOUR_ID/`
inside this same starter folder, then open its `report.ipynb` with that
extracted folder as the working directory. The notebook reuses the course
tools/public data from the starter and reads the submitted `checkpoints/`
and `evidence/` automatically. Do not include datasets or course tools in
your submission. Keep training disabled during replay.

In [1]:
from pathlib import Path
import json
import sys
import torch

HERE = Path.cwd().resolve()
PROJECT = next((p for p in [HERE, *HERE.parents] if (p / "visioncourse").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("Extract your submission inside the course starter folder, then open report.ipynb there. See START_HERE.txt.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
STUDENT = next((p for p in [HERE, *HERE.parents] if (p / "src/submission.py").is_file()
                and (p / "config.json").is_file()), PROJECT / "student_template")
PACKAGED = (STUDENT / "submission.json").is_file()
RUNS = STUDENT if PACKAGED else STUDENT / "runs/my_model"
PUBLIC = PROJECT / "release/public" if (PROJECT / "release/public").is_dir() else PROJECT / "public"
DATA = PROJECT / "release/data" if (PROJECT / "release/data").is_dir() else PROJECT / "data"
torch.set_num_threads(2)
torch.manual_seed(2026)
print({"torch": str(torch.__version__), "public_release_available": PUBLIC.is_dir(),
       "data_available": DATA.is_dir(), "packaged_report": PACKAGED, "training_started": False})

{'torch': '2.6.0+cu126', 'public_release_available': True, 'data_available': False, 'packaged_report': False, 'training_started': False}


## 2. Data

Use `instructor_dataset.zip` and the released COCO/ADE splits. Prepare them with:

```bash
python -m visioncourse.download --release public --data data --task all --classification-zip instructor_dataset.zip
```

Classification images must remain at their native **64 × 64** size during
training and validation. Horizontal flips and padded crops without rescaling
are allowed. Detection and segmentation use the fixed 320-pixel letterbox.
ADE raw label 0 is ignored; labels 1–150 map to class indices 0–149.
Detection heads return class indices 0–79, not original COCO category IDs;
`public/class_maps.json` supplies the mapping. Return boxes in the resized
320-pixel input coordinates; the evaluator restores original coordinates.

In [2]:
from visioncourse.data import ManifestDataset, find_manifest

TASKS = ("classification", "detection", "segmentation")
dataset_summary = []
for task in TASKS:
    if not PUBLIC.is_dir():
        dataset_summary.append({"task": task, "status": "Download/copy the public release first"})
        continue
    train_path = find_manifest(PUBLIC, task, "train")
    dev_path = find_manifest(PUBLIC, task, "dev")
    train_manifest = json.loads(train_path.read_text())
    dev_manifest = json.loads(dev_path.read_text())
    row = {"task": task, "train_images": len(train_manifest["samples"]),
           "development_images": len(dev_manifest["samples"])}
    first = train_manifest["samples"][0]
    if (DATA / first["image"]).is_file():
        image, target, metadata = ManifestDataset(train_path, DATA, augment=False)[0]
        row["input_tensor"] = list(image.shape)
        row["target"] = (list(target.shape) if isinstance(target, torch.Tensor)
                         else {key: list(value.shape) for key, value in target.items()})
    else:
        row["status"] = "Manifest present; prepare_data/download step is still required"
    dataset_summary.append(row)
for row in dataset_summary:
    print(row)

{'task': 'classification', 'train_images': 89953, 'development_images': 10000, 'status': 'Manifest present; prepare_data/download step is still required'}
{'task': 'detection', 'train_images': 8000, 'development_images': 1000, 'status': 'Manifest present; prepare_data/download step is still required'}
{'task': 'segmentation', 'train_images': 4000, 'development_images': 1000, 'status': 'Manifest present; prepare_data/download step is still required'}


## 3. Architecture

Modify the supplied CNN and justify your changes. Implement the backbone in
`src/backbone.py` before the source-lock deadline; its helpers are also locked.
Implement task branches in `src/heads.py`, retain the construction API in
`src/submission.py`, and update the JSON configuration to match.

`forward_features` returns `s4`, `s8`, and `s16`. Task heads receive these
features, not RGB images. The next cell checks shapes with zero inputs;
it does not measure accuracy.

In [3]:
from visioncourse.training import load_builder
from visioncourse.artifacts import parameter_counts

CONFIG_PATH = STUDENT / "config.json"
config = json.loads(CONFIG_PATH.read_text())
components = load_builder(STUDENT / "src/submission.py")(config)
for module in components.values():
    module.eval()
with torch.inference_mode():
    features64 = components["backbone"].forward_features(torch.zeros(1, 3, 64, 64))
    classification = components["classification_head"](features64)
    features320 = components["backbone"].forward_features(torch.zeros(1, 3, 320, 320))
    detection = components["detection_branch"](features320)
    segmentation = components["segmentation_branch"](features320)
assert classification.shape == (1, 200)
assert segmentation.shape == (1, 150, 320, 320)
assert len(detection) == 1 and len(detection[0]["boxes"]) <= 100
print("Classification features:", {key: list(value.shape) for key, value in features64.items()})
print("Classification logits:", list(classification.shape))
print("Detection outputs:", {key: list(value.shape) for key, value in detection[0].items()})
print("Segmentation logits:", list(segmentation.shape))
print("Learned parameters:", parameter_counts(components))

Classification features: {'s4': [1, 24, 16, 16], 's8': [1, 48, 8, 8], 's16': [1, 96, 4, 4]}
Classification logits: [1, 200]
Detection outputs: {'boxes': [0, 4], 'scores': [0], 'labels': [0]}
Segmentation logits: [1, 150, 320, 320]
Learned parameters: {'backbone': 167256, 'classification_head': 19400, 'detection_branch': 85359, 'segmentation_branch': 36390, 'total': 308405}


Before training, add an architecture diagram and explain the receptive field,
feature widths, downsampling choices, parameter budget, and changes from the
supplied CNN.

_Your analysis:_

## 4. Training

Choose epoch counts and batch sizes for your GPU budget, then set
`RUN_TRAINING=True` when ready. Justify the budget used in your experiments.

`--model-file` selects your `src/submission.py`. Development metrics select
the final checkpoints; separate resume files retain optimizer state.
Downstream runs load the selected classification checkpoint and keep its
backbone, including buffers, frozen.

In [4]:
import shlex
import subprocess

RUN_TRAINING = False
TRAIN_DEVICE = "cuda"  # Select explicitly; all notebook interface checks above run on CPU.
EPOCHS = {"classification": 10, "detection": 8, "segmentation": 8}
BATCH_SIZE = {"classification": 32, "detection": 4, "segmentation": 2}
if PACKAGED and RUN_TRAINING:
    raise ValueError("Keep RUN_TRAINING=False when replaying a final submission.")

def relative(path):
    return str(path.relative_to(PROJECT))

training_commands = []
for task in TASKS:
    command = [sys.executable, "-m", "visioncourse.training", "--task", task,
               "--release", relative(PUBLIC), "--data", relative(DATA),
               "--output", relative(RUNS), "--config", relative(STUDENT / f"configs/{task}.json"),
               "--model-file", relative(STUDENT / "src/submission.py"),
               "--epochs", str(EPOCHS[task]), "--batch-size", str(BATCH_SIZE[task]),
               "--device", TRAIN_DEVICE, "--seed", "2026"]
    if task != "classification":
        command += ["--source", relative(RUNS / "checkpoints/classification.pt")]
    training_commands.append(command)
    print(shlex.join(["python", *command[1:]]))
if RUN_TRAINING:
    for command in training_commands:
        subprocess.run(command, cwd=PROJECT, check=True)
else:
    print("Training skipped: RUN_TRAINING=False.")

python -m visioncourse.training --task classification --release public --data data --output runs/my_model --config configs/classification.json --model-file src/submission.py --epochs 10 --batch-size 32 --device cuda --seed 2026
python -m visioncourse.training --task detection --release public --data data --output runs/my_model --config configs/detection.json --model-file src/submission.py --epochs 8 --batch-size 4 --device cuda --seed 2026 --source runs/my_model/checkpoints/classification.pt
python -m visioncourse.training --task segmentation --release public --data data --output runs/my_model --config configs/segmentation.json --model-file src/submission.py --epochs 8 --batch-size 2 --device cuda --seed 2026 --source runs/my_model/checkpoints/classification.pt
Training skipped: RUN_TRAINING=False.


## 5. Source lock

Select the classification checkpoint before the week-6 lock. Use the
source-lock command in `START_HERE.txt` and submit `source_lock.json` through
the LMS for instructor registration. The digest printed below is an identity
check, not a registered receipt. After the lock, the backbone implementation,
helpers, architecture, parameters, and persistent buffers must remain unchanged.

During downstream training, keep the backbone in evaluation mode and exclude
it from the optimizer. Disabling gradients alone does not stop BatchNorm
buffer updates. The supplied utility enforces these settings and checks the
backbone state digest.

In [5]:
from visioncourse.artifacts import load_checkpoint, hash_state_dict

source_path = RUNS / "checkpoints/classification.pt"
if source_path.is_file():
    source = load_checkpoint(source_path)
    actual_digest = hash_state_dict(source["backbone"])
    assert actual_digest == source["backbone_digest"]
    components["backbone"].load_state_dict(source["backbone"], strict=True)
    components["classification_head"].load_state_dict(source["classification_head"], strict=True)
    print("Selected source digest:", actual_digest)
    for task in ("detection", "segmentation"):
        path = RUNS / f"checkpoints/{task}.pt"
        if path.is_file():
            artifact = load_checkpoint(path)
            assert artifact["backbone_digest"] == actual_digest
            components[task + "_branch"].load_state_dict(artifact[task + "_branch"], strict=True)
            print(task, "references the same frozen source")
        else:
            if PACKAGED:
                raise FileNotFoundError(f"Final submission is missing {task}.pt")
            print(task, "checkpoint not yet produced")
else:
    if PACKAGED:
        raise FileNotFoundError("Final submission is missing classification.pt")
    print("No student source checkpoint yet. Train and select one before requesting the lock receipt.")

No student source checkpoint yet. Train and select one before requesting the lock receipt.


## 6. Two controlled experiments

Run one backbone-design comparison before the source lock and one detection-head
or segmentation-decoder comparison using the locked source. In each comparison,
change one primary design choice and match the data, evaluation, compute budget,
and seed where applicable.

Complete the table from your recorded runs. Include failed runs that affect
your conclusions and retain the experiment ledger.
Use a distinct output folder for each comparison. Copy its configuration,
ledgers and summary into the final run's `evidence/comparisons/` folder;
do not overwrite earlier evidence with a later run.

| Record | Backbone comparison | Head/decoder comparison |
|---|---|---|
| Hypothesis and primary change | | |
| Control and treatment run IDs | | |
| Seed, source hash, data/config hashes | | |
| Matched budget and actual runtime | | |
| Development result and uncertainty | | |
| Interpretation and limitation | | |

In [6]:
import csv

ledger = RUNS / "evidence/experiments.csv"
if ledger.is_file():
    with ledger.open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    for row in rows[-10:]:
        print({key: row[key] for key in ("run_id", "task", "hypothesis", "optimizer_steps", "best_development_metric")})
else:
    print("No experiment records found. Run training first.")

No experiment records found. Run training first.


## 7. Learning curves and failure analysis

Plot training losses and development metrics. The next cell reads logged
losses when available.

Discuss at least one failure per task: classification errors with true and
predicted labels; missed, duplicate, or incorrect detection boxes; segmentation
boundaries and class confusions. Show successful cases for comparison, explain
how examples were selected, and label each figure with its run or checkpoint.

In [7]:
metrics_path = RUNS / "evidence/training_metrics.csv"
if metrics_path.is_file():
    import matplotlib.pyplot as plt
    with metrics_path.open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    figure, axes = plt.subplots(1, 3, figsize=(12, 3))
    for axis, task in zip(axes, TASKS):
        selected = [row for row in rows if row["task"] == task]
        run_ids = sorted({row["run_id"] for row in selected})
        for run_id in run_ids:
            run = [row for row in selected if row["run_id"] == run_id]
            axis.plot([int(row["step"]) for row in run], [float(row["train_loss"]) for row in run], label=run_id)
        axis.set(title=task, xlabel="Optimizer step", ylabel="Training loss")
        if run_ids:
            axis.legend(fontsize=7)
    figure.tight_layout()
    plt.show()
else:
    print("No training logs found.")

No training logs found.


## Final development results

Evaluate the final checkpoints using the command in `START_HERE.txt`,
then copy its `results.json` to `runs/my_model/evidence/final_development.json`.
The packaged report reads that same file from `evidence/`. Diagnostic
subset results are labelled below and must not be presented as full scores.
This cell displays saved evidence; the instructor independently reruns
inference for the official competition score.

In [8]:
final_results = RUNS / "evidence/final_development.json"
if final_results.is_file():
    results = json.loads(final_results.read_text())
    assert results["status"] == "complete"
    print("Diagnostic subset:", results.get("diagnostic", False))
    for task in TASKS:
        item = results["tasks"][task]
        print(task, item["metric"], item["score"])
    if "combined" in results:
        print("Development competition score:", results["combined"])
else:
    print("No saved final development results. Add them before final submission.")

No saved final development results. Add them before final submission.


## 8. Report and submission

Complete the architecture analysis, two comparisons, and failure analysis above.
Report classification accuracy, official COCO bbox AP, and ADE mIoU on the
released development splits. Label results from diagnostic subsets.

Include reproduction commands, seeds, dependency versions, compute use,
source-lock receipt, and final checkpoint identifiers. State the limitations
of your conclusions. Cite reused ideas and code; describe agent assistance
and your own design, debugging, and interpretation.

Run the submission preflight and save the executed `report.ipynb`. Package it
with all three checkpoints, code, configurations, and required evidence.
Exclude datasets, credentials, and training-resume files. Deadlines, scoring,
and resource limits are in `Project_1_Assignment.pdf`.

Higher competition scores rank first: 40% classification accuracy,
30% detection AP and 30% segmentation mIoU, minus the published parameter
penalty, floored at zero. All metrics are percentage points. There are
no performance thresholds or separate points rubric.